# Lesson 27 · Probability basics — roll ten thousand times and a pattern emerges from the randomness: conditional probability, independence and the law of large numbers (LLN)

**Learning goals**
1. Watch the law of large numbers in a simulation: with more samples, frequency approaches probability
2. Do simple calculations with conditional probability and independence
3. Use Bayes' theorem in a small application (such as a spam-filter intuition puzzle)

**Why it matters for Aurora**: `np.random.default_rng(seed)` runs through the whole training pipeline — batch sampling uses it to shuffle, dropout uses it to generate masks, and data augmentation uses it to pick transform parameters. With the seed fixed, results are reproducible, and only then do tuning and debugging mean anything.

← **Previous lesson** · [L26 · Calculus visualized](../3_calculus/L26_visual_calculus.ipynb)

> Last lesson covered **calculus visualized**: tangent lines, contour lines and gradient-descent trajectories, shown in pictures.
> This lesson explores **probability basics**.

## This lesson's story: why does a die "lie" while probability doesn't?

Roll a die once and you can't know what it will show. Roll it ten thousand times and the frequency of "6" settles down to 1/6 ≈ 16.67% — with the error shrinking all the while.

This is the **Law of Large Numbers (LLN)**:

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \xrightarrow{n \to \infty} \mathbb{E}[X]$$

As n grows, the sample mean ($\bar{X}_n$) approaches the population expectation. The randomness doesn't go away, but its **average effect is predictable**.

What does this have to do with Aurora?

1. **Model training**: the loss is the average of the per-sample losses in a batch. The bigger the batch, the closer that average is to the "true expected loss" — which is exactly why large-batch training is more stable.
2. **Data augmentation**: applying random pitch/noise transforms to the same audio with different seeds, the expected effect over many runs is equivalent to integrating over the whole distribution of transforms.
3. **Evaluation metrics**: a test set's WER/CER is a frequency estimate from "random sampling"; the bigger the test set, the more accurate the estimate.

In this lesson you implement `estimate_prob_six(n, seed)` to verify the LLN, and get to grips with conditional probability and Bayes' theorem — background you'll need for the cross-entropy loss in L30.

## Before we set off: five basic symbols of the probability world

The lesson is about to use a few symbols that haven't appeared before. Before getting to the law of large numbers, let's define them in the plainest words, so the formal statement won't make you nervous when you meet it.

### 1️⃣ What the summation sign ∑ means

$$\sum_{i=1}^{n} X_i$$

Nothing deep here — it just means add up the n numbers X₁, X₂, X₃ ... Xₙ:

$$\sum_{i=1}^{n} X_i = X_1 + X_2 + X_3 + \cdots + X_n$$

- the `i=1` underneath means "start from term 1"
- the `n` on top means "stop at term n"
- the `i` is a counter: "which term we're on"

**Example**: roll a die 5 times and get `[2, 6, 1, 4, 5]`. Then $\sum_{i=1}^{5} X_i = 2+6+1+4+5 = 18$; divide by 5 and you have the average, 3.6.

### 2️⃣ What is a random variable?

You've used plenty of them already — "the result of a die roll", for instance — but we've never defined one properly.

**Simply put**: a random variable is "a number that can take different values". Every time you run a random experiment (say, roll a die once), the random variable takes on one specific value.

**Examples**:
- let $X$ = "the result of one die roll" → $X$ can be 1, 2, 3, 4, 5 or 6
- let $Y$ = "the result of one coin flip" → $Y$ can be 0 (tails) or 1 (heads)
- let $Z$ = "whether a 6 came up on one roll" → $Z$ can be 0 (no 6) or 1 (a 6)

In code: the result of each run of `rng.integers(1, 7)` is one **sample** of a random variable.

### 3️⃣ What the abbreviation i.i.d. means

"i.i.d." = "independent and identically distributed". Take it in two parts:

**Independent**: the two things don't affect each other. For example:
- rolling a 6 on the first roll doesn't change the probability of rolling a 6 on the second
- your rolls have nothing to do with your neighbor David's rolls

**Identically distributed**: the rule is the same every time. For example:
- on roll 1, roll 2 and roll 100 alike, each of the numbers 1–6 has probability 1/6
- unlike a spring that goes slack after enough uses, the rule doesn't change

**Put together**: $X_1, X_2, \ldots, X_n$ are i.i.d. random variables = "every roll is independent, and every roll follows the same 1/6 rule, which never changes".

### 4️⃣ What the intersection sign ∩ means

$$A \cap B$$

means "event A **and** event B both happen".

**Example**:
- event A: "the roll is a 6"
- event B: "the roll is even"
- $A \cap B$ = "a 6 *and* even" = true only when the result is 6

In code, `(rolls == 6) & (rolls % 2 == 0)` computes exactly this intersection.

### 5️⃣ What the asymptotic notation O(...) means

$$O(1/\sqrt{n})$$

This is an "order of magnitude" notation, describing how one quantity changes as some parameter changes.

Don't be put off — put simply:
- $O(1/\sqrt{n})$ means "the same order of magnitude as $1/\sqrt{n}$; it shrinks as n grows"
- $O(n)$ means "the same order of magnitude as n; it grows linearly with n"
- $O(n^2)$ means "it grows with the square of n"

**Why use O(...) instead of the exact value?** Because mathematically we often care about "the trend", not "the exact coefficient". The error of a frequency might be $0.97 \cdot 1/\sqrt{n}$ or $1.03 \cdot 1/\sqrt{n}$, but the coefficient in front barely changes; the important information is that "the error shrinks in step with $1/\sqrt{n}$".

Now you're ready for the formal statement.

## The law of large numbers, formally

Let $X_1, X_2, \ldots, X_n$ be independent and identically distributed (i.i.d.) random variables with expectation $\mu = E[X]$. Then the sample mean

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i$$

**converges almost surely** to $\mu$ (the strong law of large numbers):

$$\bar{X}_n \xrightarrow{\text{a.s.}} \mu \quad (n \to \infty)$$

What the two pieces of notation mean:
- the arrow $\xrightarrow{\text{a.s.}}$ (a.s. = almost surely) says "when $n$ is large, $\bar{X}_n$ essentially won't stray far from $\mu$"
- the $(n \to \infty)$ on the right marks the process "as $n$ goes to infinity"

Together they say one thing: **as the sample size $n$ grows, the sample mean $\bar{X}_n$ stays ever more steadily near the theoretical expectation $\mu$**.

Put less mathematically: roll a die 10 times and the frequency of "6" might be 0.3; roll it 100 times and it might be 0.17; roll it 10000 times and it is almost certainly near 0.1667.

The code experiments below are a **visualization** of this convergence: using die rolls ($\mu = 3.5$) and the frequency of a 6 ($\mu = 1/6$) to show how $\bar{X}_n$ settles down as $n$ grows.

## 1. Checking the law of large numbers by experiment

**Why fix the seed?**

`np.random.default_rng(seed)` creates its own random-state object (the PCG64 algorithm); the same seed gives exactly the same random sequence every time. That guarantees:
- **reproducible experiments**: a classmate's code gives the same output as yours
- **traceable debugging**: when something goes wrong, you know which random sequence it was

**The LLN in numbers**:

| n (number of rolls) | frequency of "6" | order of the error |
|---|---|---|
| 10 | swings wildly, ±0.3 | O(1/√10) ≈ 0.32 |
| 1,000 | about ±0.03 | O(1/√1000) ≈ 0.032 |
| 200,000 | about ±0.001 | O(1/√200000) ≈ 0.002 |

**Why is the error O(1/√n)?** This is a core result of probability theory (a consequence of the central limit theorem). We won't derive it, but we can get a feel for it:
- Suppose you want to estimate a probability p, and you roll n times to get an estimate.
- Each roll is like an "independent signal"; as the errors (deviations) pile up, they partly cancel each other out.
- The variances of n independent signals add up (that's a property of independence), so the standard deviation of their sum grows in proportion to $\sqrt{n}$.
- The frequency is that sum divided by n, so its error is proportional to $\sqrt{n}/n = 1/\sqrt{n}$.

**The proof in numbers**: to make the estimate 10 times more precise (error from ±0.03 to ±0.003), you need to go from n = 1000 to n = 100000 — 100 times as many samples, because $\sqrt{100} = 10$. That's why machine-learning models often need huge amounts of data to train accurately.

> **Definition of conditional probability**: P(A|B) = P(A∩B) / P(B)
> **Definition of independence**: if P(A∩B) = P(A)·P(B), then A and B are independent.
> **Bayes' theorem**: P(A|B) = P(B|A)·P(A) / P(B) — exactly the probabilistic footing for "approximating the true p with a model q" in L30's cross-entropy.

## Into the lab: how probabilities emerge from samples

Watch the two parameters `size` and `seed`: `size` sets how precise the estimate is, and `seed` decides whether the result is reproducible. Run the cell below and check whether `flips.mean()` comes out close to 0.5.

## Code warm-up: how do True and False turn into a frequency?

The code ahead uses this line a lot:

```python
np.mean(rolls == 6)
```

What's happening here? Two steps:

**Step 1**: `rolls == 6` compares every element of the array with 6, giving an array of True/False. For example:
```python
rolls = np.array([1, 6, 2, 6, 3, 6])
rolls == 6  # result: [False, True, False, True, False, True]
```

**Step 2**: a neat design choice in Python and NumPy — True counts as 1 and False as 0 in arithmetic:
```python
True  == 1      # True (equivalent in arithmetic)
False == 0      # True

np.mean([False, True, False, True, False, True])
# equivalent to np.mean([0, 1, 0, 1, 0, 1])
# = (0+1+0+1+0+1) / 6 = 3/6 = 0.5
```

So `np.mean(rolls == 6)` is: how many Trues there are (that is, how many 6s), divided by the total. That is precisely the **frequency**.

It's not magic — NumPy is just cleverly designed so that boolean arrays can go straight into arithmetic.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
flips = rng.integers(0, 2, size=1000)   # 0/1 as tails/heads
print('1000 coin flips, fraction of heads ≈', flips.mean(), ' (should be close to 0.5)')

## Hands-on: small samples vs large samples

Below we compare the frequency estimate for a die at `n=10`, `100` and `10000`, and watch the estimate close in on the theoretical value 0.167 as the sample size grows.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    print(f'n={n:5d}  estimated P(rolling a 6) = {estimate:.3f}')


## Code experiment: repeating the same random experiment several times

Repeat the experiment at the same `n` with 5 different `seed`s: at `n=2000` the estimates cluster around 0.167, while at `n=20` they are clearly spread out.

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        rolls = rng.integers(1, 7, size=n)
        estimates.append(np.mean(rolls == 6))
    print(f'n={n:4d} ->', np.round(estimates, 3), 'mean=', round(float(np.mean(estimates)), 3))


## 2. ✏️ Implement `estimate_prob_six(n, seed=0)`

Simulate n rolls of a die (1–6) and return the frequency of 6.

**Line of reasoning**:
1. Create a separate random state: `rng = np.random.default_rng(seed)`, so the global random sequence stays untouched.
2. Generate all n rolls at once: `rolls = rng.integers(1, 7, size=n)`; the interval is `[1, 7)`, i.e. 1–6.
3. Count the frequency of 6: `np.mean(rolls == 6)` is equivalent to `sum(rolls == 6) / n`, and returns a float between 0 and 1.

**Reference input and output**:
- n=6, seed=0 → you get some result in which 6 shows up k times; frequency = k/6
- n=10000, seed=0 → the frequency converges to about 0.1667 (theoretical value 1/6)

**Code example** (full implementation):
```python
def estimate_prob_six(n, seed=0):
    rng = np.random.default_rng(seed)
    rolls = rng.integers(1, 7, size=n)
    return np.mean(rolls == 6)

# test
print(estimate_prob_six(10, seed=42))      # random, but reproducible
print(estimate_prob_six(10000, seed=42))   # will be close to 1/6 ≈ 0.1667
```

> 💡 Need a hint? See the full example above; or, once you've done the exercise, compare with the reference implementation in the `solutions/` folder.

### Before writing code, fill in the variable table

Before writing `estimate_prob_six`, pin down three things:
- input: `n` (number of rolls) and `seed` (random seed, default 0)
- key step: generate samples with `rng.integers(1, 7, size=n)` and take the mean of `sample == 6`
- return: a float, the frequency of 6; at `n=200000` it should be between 0.165 and 0.169

In [ ]:
def estimate_prob_six(n, seed=0):
    # ✏️ TODO: return the frequency of 6 in n die rolls
    raise NotImplementedError("Implement estimate_prob_six: generate samples with rng.integers(1,7,size=n) and return (sample==6).mean()")


In [ ]:
try:
    p = estimate_prob_six(200_000)
    print("P(rolling a 6) ≈", round(p, 4), " (true value 1/6 ≈ 0.1667)")
    assert abs(p - 1/6) < 0.01
    print("✅ Passed: with enough samples, the frequency closes in on the true probability.")
except (NotImplementedError, TypeError) as e:
    print(f"⚠️  Finish the exercise first: {e}")


**🔗 Aurora connection**: the data-augmentation pipeline and the training loop in `src/aurora/audio/` both use `np.random.default_rng(seed)` to generate batch-shuffling indices and dropout masks. With the seed fixed, two runs produce exactly the same sequence of batches — the precondition for controlled comparisons to mean anything when tuning and debugging.

## Parameter experiment: turn one knob at a time

Fix `seed=7` and raise `n` from 20 to 10000, printing the frequency each time. Notice that at n=100 the estimate wobbles by about ±0.05, and at n=10000 by about ±0.005 (standard deviation (SD) ∝ 1/√n).

In [ ]:
for n in [20, 100, 1000, 10000]:
    rng = np.random.default_rng(7)
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    error = abs(estimate - 1/6)
    print(f'n={n:5d} | estimate={estimate:.4f} | distance from theory={error:.4f}')


## 3. Conditional probability and statistical independence

### Definitions and formulas

**Conditional probability**: the probability that A happens, given that event B has happened:

$$P(A | B) = \frac{P(A \cap B)}{P(B)}$$

**In plain words**: out of all the cases where B happens, in what fraction does A happen too?

**Independence**: if `P(A | B) = P(A)` (knowing B doesn't change A's probability), then A and B are independent.

In other words: knowing that B happened leaves A's probability unchanged — B has no influence on A.

Equivalent condition: `P(A ∩ B) = P(A) · P(B)` (the multiplication rule).

**Intuition**: if A and B are independent, the probability that "both happen" is just "the product of their separate probabilities".

**Aurora connection**: dropout randomly masks neurons at every step, and the masking events of the different neurons are designed to be **independent** (mask = independent Bernoulli variables). Independence lets us use the multiplication rule to analyze the expected behavior of the whole network.

## Going deeper: deriving conditional probability

**The question**: why is `P(A|B) = P(A∩B) / P(B)`? Where does this formula come from?

**Deriving it in the language of frequencies**: imagine you roll two dice N times and record every result. Now define:
- event A: "the first die shows 6"
- event B: "the two dice sum to more than 9"

Suppose that in the N trials:
- event B happened $n_B$ times (sum of the two dice > 9)
- events A and B happened together $n_{AB}$ times (first die 6, and sum > 9)

Then **the frequency with which A also happens, given that B happens** = $\frac{n_{AB}}{n_B}$ (look only at the cases where B happened, and count how many times A happened too).

Now express the frequencies as probabilities:
$$P(A|B) = \frac{n_{AB}}{n_B} = \frac{n_{AB}/N}{n_B/N} = \frac{P(A \cap B)}{P(B)}$$

**Geometric intuition**: picture a 2-D scatter plot whose x axis is the first die and whose y axis is the second. Then:
- the set of all points is the whole sample space
- the points of event B are boxed into a particular region
- event A∩B is the points in that region that satisfy A
- $P(A|B)$ is "what fraction of the points in B's region are A points"

That is what **conditional** means: we shrink the sample space down to B, then ask what fraction of that smaller space A takes up.

In [ ]:
import numpy as np

# Simulation: roll two dice and check conditional probability
rng = np.random.default_rng(0)
n = 200_000
d1 = rng.integers(1, 7, n)
d2 = rng.integers(1, 7, n)

# A: d1=6  B: d1+d2>9
A = d1 == 6
B = (d1 + d2) > 9

p_A = A.mean()
p_B = B.mean()
p_A_given_B = (A & B).sum() / B.sum()

print(f'P(d1=6)            = {p_A:.4f}  (theory 1/6 ≈ 0.1667)')
print(f'P(d1+d2>9)         = {p_B:.4f}')
print(f'P(d1=6 | d1+d2>9) = {p_A_given_B:.4f}  (should be higher than P(d1=6))')

assert p_A_given_B > p_A, 'the conditional probability should be higher than the prior'
print('✅ P(A|B) > P(A): knowing that B happened raises the probability of A')


In [ ]:
# ✅ Checking independence numerically: two independent dice, verify the multiplication rule P(A∩B) ≈ P(A)·P(B)
rng2 = np.random.default_rng(1)
n = 200_000
die1 = rng2.integers(1, 7, n)   # first die (independent)
die2 = rng2.integers(1, 7, n)   # second die (independent)

A_indep = die1 == 6              # event A: first die shows 6
B_indep = die2 == 1              # event B: second die shows 1 (independent of the first)

p_A_i = A_indep.mean()
p_B_i = B_indep.mean()
p_AB_i = (A_indep & B_indep).mean()
product = p_A_i * p_B_i

print(f'P(A)           = {p_A_i:.4f}  (theory 1/6 ≈ 0.1667)')
print(f'P(B)           = {p_B_i:.4f}  (theory 1/6 ≈ 0.1667)')
print(f'P(A∩B)         = {p_AB_i:.5f}')
print(f'P(A)·P(B)      = {product:.5f}')
print(f'difference |P(A∩B) - P(A)·P(B)| = {abs(p_AB_i - product):.5f}')

assert abs(p_AB_i - product) < 0.001, (
    f'multiplication rule failed: |{p_AB_i:.5f} - {product:.5f}| should be < 0.001'
)
print('✅ The multiplication rule holds: the two dice are statistically independent.')


## 4. Bayes' theorem

### Derivation

Starting from the definition of conditional probability, we have two equations:

$$P(A | B) = \frac{P(A \cap B)}{P(B)}  \quad \text{...(1)}$$

$$P(B | A) = \frac{P(A \cap B)}{P(A)}  \quad \text{...(2)}$$

Both contain $P(A \cap B)$. From (1):
$$P(A \cap B) = P(A | B) \cdot P(B)$$

Substitute into the numerator of (2):
$$P(B | A) = \frac{P(A | B) \cdot P(B)}{P(A)}$$

Swap the roles of A and B and you have **Bayes' theorem**:

$$\boxed{P(A | B) = \frac{P(B | A) \cdot P(A)}{P(B)}}$$

### The terms

- $P(A)$ — **prior**: your estimate of A before the observation
- $P(B | A)$ — **likelihood**: the probability of seeing B if A is true
- $P(B)$ — **evidence**, or **marginal probability**: the "overall probability" of observing B (computed with the law of total probability)
- $P(A | B)$ — **posterior**: the updated estimate after observing B

### Likelihood vs probability

**The question**: why does Bayes' theorem call $P(B|A)$ a "likelihood" rather than a "conditional probability"?

**The answer**: the name is just convention. Strictly speaking, $P(B|A)$ really is a conditional probability. But in Bayesian analysis we emphasize a particular role it plays:
- as a "conditional probability": it tells you "given that A happened, how likely is B"
- as a "likelihood": it tells you "if I've seen the evidence B, how much does that support hypothesis A"

The higher the likelihood, the better hypothesis A explains the observed evidence B. That's why in machine learning we so often talk about "maximizing the likelihood function" — picking the parameters that make the observed data most likely.

### Spam example

Given:
- $P(\text{spam})=0.3$ (prior: the probability that an email is spam)
- $P(\text{free|spam})=0.9$ (likelihood: the probability that a spam email contains "free")
- $P(\text{free|ham})=0.01$ (likelihood: the probability that a legitimate email contains "free")

**Steps**:

1. First compute $P(\text{free})$ (the denominator) with the law of total probability:
   $$P(\text{free}) = 0.9 \times 0.3 + 0.01 \times 0.7 = 0.27 + 0.007 = 0.277$$

2. Then compute the posterior $P(\text{spam|free})$ with Bayes' theorem:
   $$P(\text{spam|free}) = \frac{P(\text{free|spam}) \times P(\text{spam})}{P(\text{free})} = \frac{0.9 \times 0.3}{0.277} = \frac{0.27}{0.277} \approx 0.975$$

**Conclusion**: prior 30% → posterior 97.5%. After observing the evidence "contains free", the belief updates dramatically — because spam is particularly fond of the word free.

## An essential tool: the law of total probability

The Bayes calculation above leaned on a more basic formula for its denominator; here it is.

**The scenario**: you want the probability that "an email contains the word free". But that event can come about along two paths:
- path 1: the email is spam, and spam contains free with high probability
- path 2: the email isn't spam (a legitimate email), but legitimate email occasionally contains free too

The **law of total probability** says: add up all the "parallel possible cases" and you get the total probability:

$$P(\text{free}) = P(\text{free|spam}) \cdot P(\text{spam}) + P(\text{free|ham}) \cdot P(\text{ham})$$

**Why addition?** Because "a spam email contains free" and "a legitimate email contains free" are **mutually exclusive** cases (an email can't be both spam and legitimate), so their probabilities simply add.

**Understanding it through frequencies**: suppose there are 1000 emails, of which 300 are spam and 700 are legitimate (ham):
- 90% of spam contains free → 300 × 0.9 = 270 emails
- 1% of legitimate email contains free → 700 × 0.01 = 7 emails
- total emails containing free → 270 + 7 = 277
- $P(\text{free}) = 277/1000 = 0.277$

That's the law of total probability in numbers.

In [ ]:
# Numerical check of Bayes' theorem
p_spam = 0.30
p_free_given_spam = 0.90
p_free_given_ham  = 0.01

p_free = p_free_given_spam * p_spam + p_free_given_ham * (1 - p_spam)
p_spam_given_free = p_free_given_spam * p_spam / p_free

print(f'P(spam)                = {p_spam:.3f}')
print(f'P(free | spam)         = {p_free_given_spam:.3f}')
print(f'P(free)                = {p_free:.3f}')
print(f'P(spam | free)         = {p_spam_given_free:.3f}')

assert abs(p_spam_given_free - 0.975) < 0.002
print('✅ Bayes\' theorem: prior 0.30 → posterior 0.975')


## Wrap-up

You can now use `estimate_prob_six(n, seed)` to return the frequency of a die face; at `n=200000` the estimate stays within about 0.01 of 0.1667. Batch shuffling and dropout-mask generation in Aurora's training loop rely on the same `default_rng` interface, and fixing the seed is the cheapest way to make experiments reproducible. The next lesson uses the mean and variance to describe the shape of a sample's distribution; today's `np.mean(sample == 6)` is a special case of the sample mean. The standard deviation of a frequency estimate shrinks roughly like `1/√n` — the key relationship when the next lesson introduces the standard deviation.

In [ ]:
# Quick check: same experiment, the bigger the sample the more stable
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.integers(1, 7, size=n)
    estimate = np.mean(samples == 6)
    print(f'n={n:4d} -> P(6)≈{estimate:.3f}')


---
⬇️ **Final check**: once you've read the wrap-up, do the whiteboard challenge below, then fill in the self-review.


## ✏️ Whiteboard challenge: probability by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: P(rolling a 6) = ? (exact fraction). Roll a die 1000 times: how many 6s do you expect?

**Question 2**: Two dice A and B are rolled independently. P(A=6) = 1/6, P(B=6) = 1/6.
P(A=6 and B=6) = ?
Independence test: does P(A∩B) = P(A)·P(B) hold?

**Question 3**: Given:
- P(spam) = 0.30
- P("free" appears | spam) = 0.90
- P("free" appears | not spam) = 0.01

Use Bayes' theorem to compute P(spam | email contains "free") = ?

**Question 4**: As the sample size n grows from 100 to 10000, the order of the frequency estimate's error goes from O(?) to O(?)?

When you've worked it out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: P(6)=1/6, expected count in 1000 rolls = 1000/6 ≈ 166.7
p_six = 1/6
expected_count = 1000 * p_six
assert np.isclose(p_six, 1/6, atol=1e-12)
assert np.isclose(expected_count, 1000/6, atol=1e-10)
print(f"Q1 ✅  P(6)={p_six:.6f}=1/6, expected count in 1000 rolls={expected_count:.1f} sixes")

# Numerical check of the LLN
try:
    p_est = estimate_prob_six(200_000, seed=42)
    assert abs(p_est - 1/6) < 0.005, f"estimate error too large: {abs(p_est-1/6):.4f}"
    print(f"     LLN check: estimate at n=200000 = {p_est:.5f}, error = {abs(p_est-1/6):.5f}")
except (NotImplementedError, TypeError):
    print("     ⬜ Implement estimate_prob_six() first, then run the answer-check cell")

# Q2: independence P(A∩B) = P(A)·P(B) = 1/36
p_ab = p_six * p_six
assert np.isclose(p_ab, 1/36, atol=1e-12)
# numerical check
rng = np.random.default_rng(99)
dice1 = rng.integers(1, 7, 200_000)
dice2 = rng.integers(1, 7, 200_000)
p_both_6 = np.mean((dice1 == 6) & (dice2 == 6))
assert abs(p_both_6 - 1/36) < 0.002
print(f"Q2 ✅  P(A=6∩B=6)=1/36={1/36:.5f}, numerical={p_both_6:.5f}, independence holds")

# Q3: Bayes' theorem
p_spam = 0.30
p_free_spam = 0.90
p_free_ham  = 0.01
p_free = p_free_spam * p_spam + p_free_ham * (1 - p_spam)
p_spam_given_free = p_free_spam * p_spam / p_free
assert np.isclose(p_free, 0.277, atol=1e-10)
assert np.isclose(p_spam_given_free, 0.9 * 0.3 / 0.277, atol=1e-10)
print(f"Q3 ✅  P('free')={p_free:.4f}, P(spam|free)={p_spam_given_free:.4f}")

# Q4: order of the error O(1/√n)
n1, n2 = 100, 10000
err1 = 1 / np.sqrt(n1)
err2 = 1 / np.sqrt(n2)
ratio = err1 / err2
assert np.isclose(ratio, 10.0, atol=1e-10)
print(f"Q4 ✅  n=100→O(1/√100)={err1:.3f}, n=10000→O(1/√10000)={err2:.3f}, error shrinks {ratio:.0f}×")
print("\n🎉 Probability whiteboard challenge passed! The LLN, conditional probability and Bayes' theorem are now in your toolkit.")

In [ ]:
# ✏️ Self-review for this lesson
l27_review = {
    "lln_statement":              None,  # can you state the law of large numbers (X̄ₙ → E[X])? True/False
    "estimate_prob_six_done":     None,  # estimate_prob_six implemented and assertions pass? True/False
    "conditional_probability":    None,  # understand P(A|B)=P(A∩B)/P(B)? True/False
    "bayes_theorem_computed":     None,  # can you compute a posterior by hand with Bayes' theorem? True/False
    "whiteboard_passed":          None,  # whiteboard challenge done on paper? True/False
}

unfilled = [k for k, v in l27_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l27_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L27 fully cleared! On to L28: mean, variance, standardization')

---

→ **Next lesson** · [L28 · Mean, variance, standardization](L28_descriptive_stats.ipynb)

> Next lesson covers **mean, variance, standardization**: descriptive statistics, z-scores and comparing distributions.